# Model 3 Training: Restormer Transformer for Image Restoration (20 Epochs - OOM-Safe)

This notebook trains Model 3: Restormer Transformer on Kaggle Dual T4 GPUs for 20 epochs with full OOM-prevention strategies.

### Why Transformer Training Crashes Without Error on Kaggle
The Restormer computes Multi-Dconv Head Transposed Self-Attention across H*W=16384 spatial tokens per image. Each batch holds thousands of intermediate attention maps in GPU memory simultaneously. When both T4 GPUs run out of VRAM, the Linux kernel OOM-killer sends SIGKILL to the notebook process which appears as a silent stop, not a Python error.

### OOM-Prevention Strategies Used in This Notebook:
- Reduced batch size to 8 (4 per GPU) with gradient accumulation over 4 steps to keep effective batch size at 32.
- Gradient checkpointing enabled in all TransformerBlock layers to recompute activations during backward pass instead of storing them.
- AMP FP16 mixed precision to cut VRAM consumption by 50 percent.
- Reduced model base dimensions (dim=32 instead of 48) and fewer blocks to reduce parameter footprint.
- Gradient clipping (max_norm=1.0) to prevent gradient explosion.
- Aggressive per-epoch GPU cache clearance and Python garbage collection.
- cuDNN autotuner enabled for fixed resolution.
- Persistent DataLoader workers with drop_last to prevent uneven batch allocations.

In [ ]:
import time
GLOBAL_START_TIME = time.time()
_cell_start = time.time()

import os
import gc
import glob
import json
import random
import zipfile
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint_utils
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Enable cuDNN autotuner for optimal kernel selection at fixed 128x128 resolution
torch.backends.cudnn.benchmark = True

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def print_timer(cell_name, cell_start):
    cell_elapsed = time.time() - cell_start
    total_elapsed = time.time() - GLOBAL_START_TIME
    print(f"[Cell Timer - {cell_name}] Cell Time: {cell_elapsed:.2f}s | Cumulative Time: {total_elapsed:.2f}s")

print_timer("Imports and Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Environment and Dataset Path Resolution
PRIMARY_DATASET_PATH = "/kaggle/input/datasets/hetulkadiya/degraded-dataset-zip-file/degraded_dataset"

def resolve_dataset_path():
    if os.path.exists(PRIMARY_DATASET_PATH):
        return PRIMARY_DATASET_PATH
    matches = glob.glob("/kaggle/input/**/degraded_dataset", recursive=True)
    if matches:
        return matches[0]
    if os.path.exists("../data/degraded_dataset"):
        return "../data/degraded_dataset"
    if os.path.exists("./degraded_dataset"):
        return "./degraded_dataset"
    return PRIMARY_DATASET_PATH

DATASET_ROOT = resolve_dataset_path()
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

# Multi-GPU Accelerator Setup
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_CUDA = False

def get_compute_device():
    global USE_CUDA
    if NUM_GPUS > 0:
        try:
            for i in range(NUM_GPUS):
                gpu_name = torch.cuda.get_device_name(i)
                total_mem = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
                print(f"Detected GPU {i}: {gpu_name} | VRAM: {total_mem:.1f} GB")
            test_tensor = torch.randn(2, 3, 16, 16, device="cuda:0")
            test_conv = nn.Conv2d(3, 16, 3, padding=1).to("cuda:0")
            _ = test_conv(test_tensor)
            del test_tensor, test_conv
            torch.cuda.empty_cache()
            USE_CUDA = True
            return torch.device("cuda")
        except Exception as e:
            print(f"Warning: CUDA error ({e}). Falling back to CPU.")
            return torch.device("cpu")
    return torch.device("cpu")

DEVICE = get_compute_device()
print(f"Active Main Device: {DEVICE} (Total GPUs Available: {NUM_GPUS})")
print(f"Dataset Root: {DATASET_ROOT}")
print(f"Output Directory: {OUTPUT_DIR}")

print_timer("Path and Multi-GPU Device Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Dataset Definition for 7-Combination Paired Degraded Images
class KaggleDegradedDataset(Dataset):
    def __init__(self, root_dir, split="train", target_size=(128, 128)):
        self.root_dir = root_dir
        self.split = split
        self.target_size = target_size
        self.split_dir = os.path.join(root_dir, split)
        self.clean_dir = os.path.join(self.split_dir, "clean")
        self.combos = [
            "c1_noise_lowlight",
            "c2_noise_blur",
            "c3_lowlight_blur",
            "c4_noise_only",
            "c5_lowlight_only",
            "c6_blur_only",
            "c7_all_three"
        ]
        self.pairs = []
        if os.path.exists(self.clean_dir):
            clean_files = sorted(
                glob.glob(os.path.join(self.clean_dir, "*.png")) +
                glob.glob(os.path.join(self.clean_dir, "*.jpg"))
            )
            for clean_path in clean_files:
                filename = os.path.basename(clean_path)
                for combo in self.combos:
                    deg_path = os.path.join(self.split_dir, combo, filename)
                    if os.path.exists(deg_path):
                        self.pairs.append((deg_path, clean_path, combo))
        self.to_tensor = T.Compose([
            T.Resize(target_size),
            T.ToTensor()
        ])
        print(f"Loaded [{split.upper()}] dataset split: {len(self.pairs)} paired samples.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        deg_path, clean_path, combo = self.pairs[idx]
        with Image.open(deg_path) as deg_img, Image.open(clean_path) as clean_img:
            deg_tensor = self.to_tensor(deg_img.convert("RGB"))
            clean_tensor = self.to_tensor(clean_img.convert("RGB"))
        return deg_tensor, clean_tensor

# OOM-safe batch size: Small physical batch with gradient accumulation to keep
# effective batch size at 32. Transformer attention maps need much less VRAM this way.
PHYSICAL_BATCH = 8          # actual batches loaded per step (4 per GPU)
ACCUM_STEPS = 4             # accumulate 4 steps -> effective batch = 8 * 4 = 32
NUM_WORKERS = 2 if USE_CUDA else 0

train_dataset = KaggleDegradedDataset(DATASET_ROOT, split="train")
val_dataset   = KaggleDegradedDataset(DATASET_ROOT, split="val")
test_dataset  = KaggleDegradedDataset(DATASET_ROOT, split="test")

loader_kwargs = {
    "num_workers":        NUM_WORKERS,
    "pin_memory":         USE_CUDA,
    "persistent_workers": (NUM_WORKERS > 0)
}

train_loader = DataLoader(train_dataset, batch_size=PHYSICAL_BATCH, shuffle=True,  drop_last=True,  **loader_kwargs)
val_loader   = DataLoader(val_dataset,   batch_size=PHYSICAL_BATCH, shuffle=False, drop_last=False, **loader_kwargs)
test_loader  = DataLoader(test_dataset,  batch_size=PHYSICAL_BATCH, shuffle=False, drop_last=False, **loader_kwargs)

print(f"Physical Batch Size: {PHYSICAL_BATCH} | Gradient Accumulation Steps: {ACCUM_STEPS} | Effective Batch: {PHYSICAL_BATCH * ACCUM_STEPS}")
print(f"DataLoader Workers: {NUM_WORKERS} | Persistent Workers: {NUM_WORKERS > 0}")
print(f"Dataset Pair Counts -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print_timer("Dataset and DataLoaders Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Model 3: Restormer Transformer with Gradient Checkpointing
# Gradient checkpointing trades compute for memory: instead of storing all
# intermediate attention activations, they are recomputed during the backward pass.
# This reduces peak VRAM usage by ~40-60% with only ~20% compute overhead.

class MDTA(nn.Module):
    """Multi-Dconv Head Transposed Self-Attention (MDTA) module."""
    def __init__(self, dim, num_heads=4, bias=False):
        super(MDTA, self).__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1, bias=bias)
        self.qkv_dwconv = nn.Conv2d(dim * 3, dim * 3, kernel_size=3, stride=1, padding=1, groups=dim * 3, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)
        q = q.reshape(b, self.num_heads, -1, h * w)
        k = k.reshape(b, self.num_heads, -1, h * w)
        v = v.reshape(b, self.num_heads, -1, h * w)
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        out = (attn @ v).reshape(b, c, h, w)
        out = self.project_out(out)
        return out

class GDFN(nn.Module):
    """Gated-Dconv Feed-Forward Network (GDFN) module."""
    def __init__(self, dim, ffn_expansion_factor=2.66, bias=False):
        super(GDFN, self).__init__()
        hidden_features = int(dim * ffn_expansion_factor)
        self.project_in  = nn.Conv2d(dim, hidden_features * 2, kernel_size=1, bias=bias)
        self.dwconv      = nn.Conv2d(hidden_features * 2, hidden_features * 2, kernel_size=3, stride=1, padding=1, groups=hidden_features * 2, bias=bias)
        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x = F.gelu(x1) * x2
        x = self.project_out(x)
        return x

class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super(LayerNorm2d, self).__init__()
        self.weight = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.bias   = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.eps    = eps

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight * x + self.bias

class TransformerBlock(nn.Module):
    """TransformerBlock with optional gradient checkpointing to save VRAM."""
    def __init__(self, dim, num_heads=4, ffn_expansion_factor=2.66, bias=False, use_checkpoint=True):
        super(TransformerBlock, self).__init__()
        self.norm1 = LayerNorm2d(dim)
        self.attn  = MDTA(dim, num_heads, bias)
        self.norm2 = LayerNorm2d(dim)
        self.ffn   = GDFN(dim, ffn_expansion_factor, bias)
        self.use_checkpoint = use_checkpoint

    def _inner_forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

    def forward(self, x):
        if self.use_checkpoint and x.requires_grad:
            return checkpoint_utils.checkpoint(self._inner_forward, x, use_reentrant=False)
        return self._inner_forward(x)

class Restormer(nn.Module):
    """
    Model 3: Restormer Transformer with reduced dim=32 and gradient checkpointing.
    dim reduced from 48 to 32 to cut VRAM footprint by ~55% while retaining model quality.
    """
    def __init__(self, in_channels=3, out_channels=3, dim=32, num_blocks=[2, 2, 2, 2], heads=[1, 2, 4, 4]):
        super(Restormer, self).__init__()
        self.patch_embed = nn.Conv2d(in_channels, dim, kernel_size=3, padding=1)

        # Encoder Level 1
        self.encoder_level1 = nn.Sequential(*[
            TransformerBlock(dim=dim, num_heads=heads[0], use_checkpoint=True)
            for _ in range(num_blocks[0])
        ])
        self.down1_2 = nn.Conv2d(dim, dim * 2, kernel_size=4, stride=2, padding=1)

        # Encoder Level 2
        self.encoder_level2 = nn.Sequential(*[
            TransformerBlock(dim=dim * 2, num_heads=heads[1], use_checkpoint=True)
            for _ in range(num_blocks[1])
        ])

        # Bottleneck
        self.bottleneck = nn.Sequential(*[
            TransformerBlock(dim=dim * 2, num_heads=heads[2], use_checkpoint=True)
            for _ in range(num_blocks[2])
        ])

        # Decoder Level 1
        self.up2_1      = nn.ConvTranspose2d(dim * 2, dim, kernel_size=4, stride=2, padding=1)
        self.reduce_chan = nn.Conv2d(dim * 2, dim, kernel_size=1)
        self.decoder_level1 = nn.Sequential(*[
            TransformerBlock(dim=dim, num_heads=heads[3], use_checkpoint=True)
            for _ in range(num_blocks[3])
        ])

        self.output = nn.Sequential(
            nn.Conv2d(dim, out_channels, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, inp_img):
        feat = self.patch_embed(inp_img)
        e1   = self.encoder_level1(feat)
        d1   = self.down1_2(e1)
        e2   = self.encoder_level2(d1)
        b    = self.bottleneck(e2)
        u1   = self.up2_1(b)
        u1   = self.reduce_chan(torch.cat([u1, e1], dim=1))
        d_out = self.decoder_level1(u1)
        out  = self.output(d_out + feat)
        return out

base_model = Restormer()

# Wrap model with DataParallel if multiple GPUs are detected
if NUM_GPUS > 1:
    print(f"Enabling Multi-GPU DataParallel across {NUM_GPUS} GPUs!")
    model = nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(DEVICE)

num_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"Restormer Transformer Model initialized. Total Trainable Parameters: {num_params:,}")

# Report initial GPU memory usage
if USE_CUDA:
    alloc = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    print(f"Initial GPU Memory -> Allocated: {alloc:.2f} GB | Reserved: {reserved:.2f} GB")

print_timer("Multi-GPU Transformer Model Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Loss Function and Metric Evaluation Functions (PSNR and SSIM)
class CombinedLoss(nn.Module):
    def __init__(self, l1_weight=0.8, mse_weight=0.2):
        super(CombinedLoss, self).__init__()
        self.l1_weight  = l1_weight
        self.mse_weight = mse_weight
        self.l1  = nn.L1Loss()
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        return self.l1_weight * self.l1(pred, target) + self.mse_weight * self.mse(pred, target)

def calculate_batch_psnr(pred, target):
    with torch.no_grad():
        pred_f   = pred.float()
        target_f = target.float()
        mse = torch.mean((pred_f - target_f) ** 2, dim=[1, 2, 3])
        psnr = 10.0 * torch.log10(1.0 / (mse + 1e-8))
        return torch.mean(psnr).item()

def calculate_batch_ssim(pred, target):
    with torch.no_grad():
        pred_f   = pred.float()
        target_f = target.float()
        mu_x  = F.avg_pool2d(pred_f,   3, 1, 1)
        mu_y  = F.avg_pool2d(target_f, 3, 1, 1)
        sigma_x  = F.avg_pool2d(pred_f   * pred_f,   3, 1, 1) - mu_x ** 2
        sigma_y  = F.avg_pool2d(target_f * target_f, 3, 1, 1) - mu_y ** 2
        sigma_xy = F.avg_pool2d(pred_f   * target_f, 3, 1, 1) - mu_x * mu_y
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / \
                   ((mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x + sigma_y + C2))
        return torch.mean(ssim_map).item()

def extract_clean_state_dict(m):
    return m.module.state_dict() if isinstance(m, nn.DataParallel) else m.state_dict()

def load_clean_state_dict(m, path, device):
    state_dict = torch.load(path, map_location=device)
    if isinstance(m, nn.DataParallel):
        m.module.load_state_dict(state_dict)
    else:
        m.load_state_dict(state_dict)

criterion = CombinedLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.999), weight_decay=1e-5)

EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# AMP GradScaler: automatically handles FP16 underflow by scaling gradients
scaler = torch.cuda.amp.GradScaler(enabled=USE_CUDA)

print(f"Epochs: {EPOCHS} | Optimizer: Adam lr=1e-4 | Scheduler: CosineAnnealing | AMP: {USE_CUDA}")
print(f"Gradient Accumulation: {ACCUM_STEPS} steps | Effective Batch: {PHYSICAL_BATCH * ACCUM_STEPS}")
print_timer("Loss and Metric Functions Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# 20-Epoch OOM-Safe Training and Validation Loop with Gradient Accumulation
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "transformer_model.pth")
best_val_psnr = 0.0

history = {
    "train_loss": [],
    "val_loss":   [],
    "val_psnr":   [],
    "val_ssim":   [],
    "lr":         []
}

print(f"Starting OOM-Safe Multi-GPU Training for {EPOCHS} Epochs on {DEVICE}...")

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # ---- Training Phase ----
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    train_pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                      desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]", leave=False, mininterval=2.0)

    for step_idx, (degraded, clean) in train_pbar:
        degraded = degraded.to(DEVICE, non_blocking=True)
        clean    = clean.to(DEVICE, non_blocking=True)

        # Forward pass with AMP
        with torch.cuda.amp.autocast(enabled=USE_CUDA, dtype=torch.float16):
            output = model(degraded)
            # Divide loss by accumulation steps to normalize the gradient
            loss = criterion(output, clean) / ACCUM_STEPS

        scaler.scale(loss).backward()

        # Optimizer step every ACCUM_STEPS micro-batches
        if (step_idx + 1) % ACCUM_STEPS == 0 or (step_idx + 1) == len(train_loader):
            # Gradient clipping to prevent gradient explosion
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # Multiply back to get true loss value for logging
        batch_loss = loss.item() * ACCUM_STEPS
        train_loss += batch_loss
        train_pbar.set_postfix({"Loss": f"{batch_loss:.4f}"})

        # Free intermediate tensors every step
        del degraded, clean, output, loss

    train_loss /= max(len(train_loader), 1)
    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(round(train_loss, 4))
    history["lr"].append(round(current_lr, 8))

    # ---- Validation Phase ----
    model.eval()
    val_loss = 0.0
    val_psnr = 0.0
    val_ssim = 0.0

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Val]  ",
                    leave=False, mininterval=2.0)

    with torch.no_grad():
        for degraded, clean in val_pbar:
            degraded = degraded.to(DEVICE, non_blocking=True)
            clean    = clean.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=USE_CUDA, dtype=torch.float16):
                restored = model(degraded)
                loss = criterion(restored, clean)
            val_loss += loss.item()
            val_psnr += calculate_batch_psnr(restored, clean)
            val_ssim += calculate_batch_ssim(restored, clean)
            del degraded, clean, restored, loss

    val_loss /= max(len(val_loader), 1)
    val_psnr /= max(len(val_loader), 1)
    val_ssim /= max(len(val_loader), 1)

    history["val_loss"].append(round(val_loss, 4))
    history["val_psnr"].append(round(val_psnr, 2))
    history["val_ssim"].append(round(val_ssim, 4))

    # Cosine Annealing step
    scheduler.step()

    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} [{epoch_time:.1f}s] -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f} | LR: {current_lr:.2e}")

    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        clean_sd = extract_clean_state_dict(model)
        torch.save(clean_sd, MODEL_SAVE_PATH)
        print(f"  --> Saved best checkpoint to '{MODEL_SAVE_PATH}' (Val PSNR: {val_psnr:.2f} dB)")

    # Aggressive GPU memory cleanup after every epoch
    if USE_CUDA:
        alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        torch.cuda.empty_cache()
        print(f"  [Memory] GPU Allocated before cleanup: {alloc:.2f} GB | Cache cleared.")
    gc.collect()

print_timer("20-Epoch OOM-Safe Multi-GPU Training Loop", _cell_start)

In [ ]:
_cell_start = time.time()

# Final Evaluation and Benchmark on Test Set
if os.path.exists(MODEL_SAVE_PATH):
    load_clean_state_dict(model, MODEL_SAVE_PATH, DEVICE)

model.eval()
deg_psnr_total, deg_ssim_total = 0.0, 0.0
res_psnr_total, res_ssim_total = 0.0, 0.0

with torch.no_grad():
    for degraded, clean in tqdm(test_loader, desc="Evaluating Test Set", mininterval=1.0):
        degraded = degraded.to(DEVICE, non_blocking=True)
        clean    = clean.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_CUDA, dtype=torch.float16):
            restored = model(degraded)
        deg_psnr_total += calculate_batch_psnr(degraded, clean)
        deg_ssim_total += calculate_batch_ssim(degraded, clean)
        res_psnr_total += calculate_batch_psnr(restored, clean)
        res_ssim_total += calculate_batch_ssim(restored, clean)
        del degraded, clean, restored

n_test_batches = max(len(test_loader), 1)
avg_deg_psnr = deg_psnr_total / n_test_batches
avg_deg_ssim = deg_ssim_total / n_test_batches
avg_res_psnr = res_psnr_total / n_test_batches
avg_res_ssim = res_ssim_total / n_test_batches

psnr_gain = avg_res_psnr - avg_deg_psnr
ssim_gain = avg_res_ssim - avg_deg_ssim

print("\n======================================================")
print("MODEL 3 (RESTORMER TRANSFORMER - 20 EPOCHS) TEST BENCHMARK")
print("======================================================")
print(f"GPUs Utilized           : {max(NUM_GPUS, 1)}")
print(f"Degraded Input Quality  : PSNR = {avg_deg_psnr:.2f} dB | SSIM = {avg_deg_ssim:.4f}")
print(f"Transformer Restored    : PSNR = {avg_res_psnr:.2f} dB | SSIM = {avg_res_ssim:.4f}")
print(f"Restoration dB Gain     : PSNR Gain = +{psnr_gain:.2f} dB | SSIM Gain = +{ssim_gain:.4f}")
print("======================================================")

metrics_json_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
metrics_data = {
    "model_name": "Restormer Transformer Restoration Network (20 Epochs, OOM-Safe)",
    "gpus_used": max(NUM_GPUS, 1),
    "epochs": EPOCHS,
    "physical_batch_size": PHYSICAL_BATCH,
    "gradient_accumulation_steps": ACCUM_STEPS,
    "effective_batch_size": PHYSICAL_BATCH * ACCUM_STEPS,
    "loss_functions": ["L1Loss", "MSELoss"],
    "optimizer": "Adam (lr=1e-4, CosineAnnealing, AMP-FP16, GradCheckpoint)",
    "history": history,
    "test_results": {
        "degraded_psnr": round(avg_deg_psnr, 2),
        "degraded_ssim": round(avg_deg_ssim, 4),
        "restored_psnr": round(avg_res_psnr, 2),
        "restored_ssim": round(avg_res_ssim, 4),
        "psnr_gain_db":  round(psnr_gain, 2),
        "ssim_gain":     round(ssim_gain, 4)
    }
}
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)

print_timer("Test Evaluation and Metrics Export", _cell_start)

In [ ]:
_cell_start = time.time()

# Plot Training Curves and Generate Sample Visual Restorations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss Curve
axes[0].plot(range(1, EPOCHS + 1), history["train_loss"], label="Train Loss", marker="o", markersize=4, color="#ef4444")
axes[0].plot(range(1, EPOCHS + 1), history["val_loss"],   label="Val Loss",   marker="s", markersize=4, color="#3b82f6")
axes[0].set_title("L1 + MSE Loss Curve (20 Epochs)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# PSNR Curve
axes[1].plot(range(1, EPOCHS + 1), history["val_psnr"], label="Val PSNR (dB)", marker="^", markersize=4, color="#10b981")
axes[1].set_title("Validation PSNR Curve (dB)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

# SSIM Curve
axes[2].plot(range(1, EPOCHS + 1), history["val_ssim"], label="Val SSIM", marker="d", markersize=4, color="#8b5cf6")
axes[2].set_title("Validation SSIM Score Curve", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("SSIM Score")
axes[2].grid(True, linestyle="--", alpha=0.5)
axes[2].legend()

plt.tight_layout()
plots_image_path = os.path.join(OUTPUT_DIR, "loss_psnr_ssim_curves.png")
plt.savefig(plots_image_path, dpi=300)
plt.show()

# Visual Restoration Samples Plot
model.eval()
for deg_b, clean_b in test_loader:
    deg_b   = deg_b.to(DEVICE)
    clean_b = clean_b.to(DEVICE)
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=USE_CUDA, dtype=torch.float16):
            res_b = model(deg_b)
    break

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(min(3, deg_b.shape[0])):
    deg_np   = deg_b[i].cpu().float().permute(1, 2, 0).numpy()
    res_np   = res_b[i].cpu().float().permute(1, 2, 0).numpy()
    clean_np = clean_b[i].cpu().float().permute(1, 2, 0).numpy()

    axes[i, 0].imshow(clean_np)
    axes[i, 0].set_title("Original Clean", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(deg_np)
    axes[i, 1].set_title("Degraded Input", fontsize=9)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(res_np)
    axes[i, 2].set_title("Transformer Restored", fontsize=9)
    axes[i, 2].axis("off")

plt.suptitle("Restormer Transformer Test Sample Restorations (20 Epochs)", fontsize=13)
plt.tight_layout()
samples_image_path = os.path.join(OUTPUT_DIR, "test_restoration_samples.png")
plt.savefig(samples_image_path, dpi=300)
plt.show()

print_timer("Plots and Visual Restorations", _cell_start)

In [ ]:
_cell_start = time.time()

# Package Checkpoints, Metrics, Plots into transformer_restoration_results.zip
ZIP_FILE_PATH = os.path.join(OUTPUT_DIR, "transformer_restoration_results.zip")

files_to_zip = [
    (MODEL_SAVE_PATH,    "transformer_model.pth"),
    (metrics_json_path,  "training_metrics.json"),
    (plots_image_path,   "loss_psnr_ssim_curves.png"),
    (samples_image_path, "test_restoration_samples.png")
]

with zipfile.ZipFile(ZIP_FILE_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for src_path, arcname in files_to_zip:
        if os.path.exists(src_path):
            zipf.write(src_path, arcname=arcname)
            print(f"Added to Zip: {arcname}")

print("\n======================================================")
print("RESULTS ZIP ARCHIVE CREATED SUCCESSFULLY")
print(f"Zip Path: {ZIP_FILE_PATH}")
print(f"Size: {os.path.getsize(ZIP_FILE_PATH) / (1024 * 1024):.2f} MB")
print("Contains: transformer_model.pth, training_metrics.json, curves, visual samples")
print("======================================================")

print_timer("Zip Packaging and Export", _cell_start)